# Análise de Resultados - PUMA FL
Este notebook processa as múltiplas simulações (runs) geradas pelo servidor e plota as curvas de performance comparando diferentes estratégias, exibindo o intervalo de confiança (desvio padrão).

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# Configuração de estilo para gráficos nível artigo acadêmico
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'legend.fontsize': 14,
    'lines.linewidth': 2.5
})

# Caminho padrão para a pasta de resultados (ajuste se necessário)
RESULTS_DIR = "../results/"

In [ ]:
def load_strategy_results(dataset, strategy, algo="FedAVG", prune="withou_Prune"):
    """
    Busca os arquivos h5 baseados na estratégia e calcula média e desvio padrão.
    """
    pattern = os.path.join(RESULTS_DIR, f"{dataset}_{strategy}_{prune}_{algo}_run*.h5")
    files = glob.glob(pattern)
    
    if not files:
        print(f"⚠️ Nenhum arquivo encontrado para: {pattern}")
        return None
        
    print(f"✅ Encontrados {len(files)} arquivos para a estratégia '{strategy}'")
    
    acc_runs, loss_runs, mb_runs, mb_bruto_runs, time_runs = [], [], [], [], []
    
    for f in files:
        with h5py.File(f, 'r') as hf:
            acc_runs.append(np.array(hf['rs_test_acc']))
            loss_runs.append(np.array(hf['rs_train_loss']))
            mb_runs.append(np.array(hf['sended_model_Mb']))
            mb_bruto_runs.append(np.array(hf['Sended_without_quant']))
            time_runs.append(np.array(hf['Round_time']))
            
    return {
        'acc_mean': np.mean(acc_runs, axis=0), 'acc_std': np.std(acc_runs, axis=0),
        'loss_mean': np.mean(loss_runs, axis=0), 'loss_std': np.std(loss_runs, axis=0),
        'mb_mean': np.mean(mb_runs, axis=0), 'mb_std': np.std(mb_runs, axis=0),
        'mb_bruto_mean': np.mean(mb_bruto_runs, axis=0), 'mb_bruto_std': np.std(mb_bruto_runs, axis=0),
        'time_mean': np.mean(time_runs, axis=0), 'time_std': np.std(time_runs, axis=0),
    }

In [ ]:
# === CARREGAMENTO DOS DADOS ===
DATASET = "MNIST"  # Altere para "OxfordPets", "Cifar100", etc.

print("Carregando resultados...")

# Ajuste os parâmetros de acordo com o que você rodou no run.sh
dados_estrat_1 = load_strategy_results(DATASET, strategy="cnn", algo="FedALA", prune="withou_Prune")

# Exemplo de como carregar a segunda estratégia para comparar:
# dados_estrat_2 = load_strategy_results(DATASET, strategy="lora", algo="FedALA", prune="withou_Prune")

### 1. Desempenho do Modelo (Acurácia Global)

In [ ]:
plt.figure(figsize=(10, 6))

if dados_estrat_1:
    plt.plot(dados_estrat_1['acc_mean'], label='Estratégia 1 (CNN)', color='#1f77b4')
    plt.fill_between(range(len(dados_estrat_1['acc_mean'])), 
                     dados_estrat_1['acc_mean'] - dados_estrat_1['acc_std'], 
                     dados_estrat_1['acc_mean'] + dados_estrat_1['acc_std'], color='#1f77b4', alpha=0.15)

# Descomente para plotar a comparação quando tiver os dados
# if dados_estrat_2:
#     plt.plot(dados_estrat_2['acc_mean'], label='Estratégia 2 (LoRA)', color='#d62728')
#     plt.fill_between(range(len(dados_estrat_2['acc_mean'])), 
#                      dados_estrat_2['acc_mean'] - dados_estrat_2['acc_std'], 
#                      dados_estrat_2['acc_mean'] + dados_estrat_2['acc_std'], color='#d62728', alpha=0.15)

plt.title(f"Acurácia Global - {DATASET}")
plt.xlabel("Rodadas (Rounds)")
plt.ylabel("Acurácia (%)")
plt.legend()
plt.tight_layout()
plt.show()

### 2. Custo de Comunicação Acumulado (Megabytes)

In [ ]:
plt.figure(figsize=(10, 6))

if dados_estrat_1:
    eixo_x = range(len(dados_estrat_1['mb_mean']))
    
    # Linha mostrando o tamanho se enviasse o modelo bruto (Sem compressão)
    plt.plot(eixo_x, dados_estrat_1['mb_bruto_mean'], label='Tamanho Bruto Padrão', color='gray', linestyle='--')
    
    # Linha mostrando o que foi efetivamente trafegado com o seu Framework
    plt.plot(eixo_x, dados_estrat_1['mb_mean'], label='Trafegado (Estratégia 1)', color='#2ca02c')
    plt.fill_between(eixo_x, 
                     dados_estrat_1['mb_mean'] - dados_estrat_1['mb_std'], 
                     dados_estrat_1['mb_mean'] + dados_estrat_1['mb_std'], color='#2ca02c', alpha=0.15)

plt.title(f"Custo de Rede Acumulado - {DATASET}")
plt.xlabel("Rodadas (Rounds)")
plt.ylabel("Megabytes (MB)")
plt.legend()
plt.tight_layout()
plt.show()

### 3. Latência de Treinamento (Tempo)

In [ ]:
plt.figure(figsize=(10, 4))

if dados_estrat_1:
    plt.plot(dados_estrat_1['time_mean'], label='Tempo (Estratégia 1)', color='#ff7f0e', marker='o')
    plt.fill_between(range(len(dados_estrat_1['time_mean'])), 
                     dados_estrat_1['time_mean'] - dados_estrat_1['time_std'], 
                     dados_estrat_1['time_mean'] + dados_estrat_1['time_std'], color='#ff7f0e', alpha=0.15)

plt.title("Tempo de Execução por Rodada")
plt.xlabel("Rodadas (Rounds)")
plt.ylabel("Tempo (Segundos)")
plt.legend()
plt.tight_layout()
plt.show()